# AoC 2024 Day 18 — RAM Run

**Python — BFS with the grid scale as a parameter**

Puzzle: <https://adventofcode.com/2024/day/18>

---

> **On puzzle text and inputs.** Advent of Code is Eric Wastl's work, and he asks that puzzle text and per-user inputs not be redistributed. So this notebook carries a summary in my own words plus the *published* example, and pulls the real input at runtime from a local cache that is gitignored. Read the puzzle at the link above.

> **Part 1 only.** Advent of Code reveals Part Two only after a correct Part One submission, and this day is unsolved — so part 2's text does not exist to work from yet. Submitting the answer below unlocks it.

---

## The puzzle

A list of `x,y` coordinates — bytes falling one per nanosecond into a square memory grid, in the order given. A fallen byte **corrupts** its coordinate; corrupted cells cannot be entered.

You start at the top-left `(0,0)` and want the exit at the bottom-right corner. Moves are up/down/left/right, one cell at a time, and you cannot leave the grid.

- **Part 1** — let the first *N* bytes fall, then find the minimum number of steps from corner to corner.

The two scales matter: the published example is a **7×7 grid after 12 bytes** (answer 22), while the real input is a **71×71 grid after 1024 bytes**. Same format, different constants.

## The approach

Breadth-first search on a 71×71 grid is ~5000 cells and ~20000 edge checks. CPython walks the whole thing in a couple of milliseconds using a `deque` and a `set`.

The Spark framing is the same iterative-join shape as day 16, minus the priority queue: hold the current frontier as a DataFrame, cross-join it to a 4-row steps table, anti-join against `seen` to drop revisits, union into `seen`, repeat. The shortest path is 22 steps in the example and several hundred on the real grid — **that is one Spark job per BFS level**, each shuffling a frontier of at most a few dozen rows. The frontier is far too small to amortise a shuffle, and the level count is exactly what you cannot parallelise away: level *k+1* is undefined until level *k* is complete.

The design decision actually worth studying here is the **signature**:

```python
def part1(spark, data, size: int = 71, fallen: int = 1024) -> int:
```

The published example is a 7×7 grid after 12 bytes; the real input is 71×71 after 1024. Those are two different problem scales over the same input format. Hardcoding the real values would make the example untestable, and hardcoding the example values would make the answer wrong — so both become parameters with real-input defaults, and the test passes `size=7, fallen=12`. Note this is not the same as `size` being derivable from the data: the grid extends to coordinates no byte ever lands on, so it genuinely cannot be inferred.

## Setup

Connect to the cluster's Spark Connect endpoint and import the solution.

In [ ]:
import sys

sys.path.insert(0, '..')  # so `aoc_spark` resolves when running from notebooks/

from aoc_spark.session import get_spark
from aoc_spark.inputs import get_input
from aoc_spark.y2024 import day18

spark = get_spark('aoc-2024-day18')
print('Spark', spark.version)

## The published example

The same data the test suite asserts on.

The example runs at a different scale than the real input, so `fallen=12`, `size=7` are passed explicitly rather than hardcoded.

In [ ]:
EXAMPLE = '5,4\n4,2\n4,5\n3,0\n2,1\n6,3\n2,4\n1,5\n0,6\n3,3\n2,6\n5,1\n1,2\n5,5\n2,5\n6,5\n1,4\n0,4\n6,4\n1,1\n6,1\n1,0\n0,5\n1,6\n2,0\n'

print('part 1:', day18.part1(spark, EXAMPLE, fallen=12, size=7), '(expected 22)')

### The distance field, and the levels it took

The grid below shows each reachable cell labelled with its BFS distance (mod 10), `#` for corrupted, `.` for unreachable. Underneath is the frontier size at each level — that list *is* the sequential dependency, and its length is how many Spark jobs a distributed version would submit.

In [ ]:
from collections import deque

SIZE, FALLEN = 7, 12  # the example scale; the real input is 71, 1024

coords = [tuple(int(v) for v in line.split(',')) for line in EXAMPLE.strip().splitlines()]
corrupted = set(coords[:FALLEN])
print(f'{len(coords)} bytes listed; only the first {FALLEN} have fallen yet')

# Same BFS as part1, but recording the distance of every cell it reaches.
goal = (SIZE - 1, SIZE - 1)
dist = {(0, 0): 0}
queue = deque([(0, 0)])
levels = []
while queue:
    levels.append(len(queue))
    for _ in range(len(queue)):  # one BFS level = one Spark job
        x, y = queue.popleft()
        for dx, dy in day18.STEPS:
            nxt = (x + dx, y + dy)
            if not (0 <= nxt[0] < SIZE and 0 <= nxt[1] < SIZE):
                continue
            if nxt in corrupted or nxt in dist:
                continue
            dist[nxt] = dist[(x, y)] + 1
            queue.append(nxt)

# The grid, with each reachable cell showing its distance mod 10.
for y in range(SIZE):
    row = []
    for x in range(SIZE):
        if (x, y) in corrupted:
            row.append('#')
        elif (x, y) in dist:
            row.append(str(dist[(x, y)] % 10))
        else:
            row.append('.')
    print(' '.join(row))

print()
print(f'reached the exit {goal} in {dist[goal]} steps, visiting {len(dist)} of {SIZE * SIZE} cells')
print(f'BFS levels (frontier sizes): {levels}')
print(f'-> {len(levels)} sequential rounds, largest frontier {max(levels)} cells')

## The real input

`get_input` is cache-first: local gitignored file → Postgres → adventofcode.com. In practice it hits the local file and never touches the network.

In [ ]:
import time

data = get_input(2024, 18)
print(f'input: {len(data):,} chars, {len(data.splitlines()):,} lines')

started = time.perf_counter()
answer = day18.part1(spark, data)
print(f'part 1: {answer}  ({(time.perf_counter() - started) * 1000:.0f} ms)')

## Cross-check

Days 6+ have no known-good answer, so correctness rests on an independently written plain-Python implementation agreeing with the Spark one. That is evidence, not proof — a shared misreading of the puzzle would survive both.

In [ ]:
from reference_python.y2024 import day18 as reference

cross = reference.part1(data)
print('reference:', cross)
print('agree:    ', cross == answer)

## Notes & gotchas

- **`size` and `fallen` are parameters, not constants.** The example is 7×7 after 12 bytes; the real input is 71×71 after 1024. Defaults are the real-input values, and the notebook/test pass `size=7, fallen=12` for the example. Get these backwards and you get a plausible number that is simply wrong.
- `size` is the **side length**, so valid coordinates are `0..size-1` and the goal is `(size-1, size-1)` — `70,70` for the real input, not `71,71`. The puzzle text states the range as `0 to 70`, which is the off-by-one waiting to happen.
- Coordinates are **`x,y` = column,row**, not row,col. It happens not to matter here because the grid is square and start/goal are both on the diagonal, so a transposed reading gives the same answer — which is exactly the kind of luck that hides the bug until part 2 or a non-square grid.
- `seen` is marked at **enqueue** time, not dequeue. Marking on dequeue lets the same cell enter the queue several times before it is first processed; the answer stays correct but the queue bloats.
- The input lists more bytes than `fallen`; the slice `[:fallen]` is what makes the rest invisible. The remaining bytes are not junk — they are there for part 2.
- Unreachable goal returns `-1` rather than raising. With the real defaults the grid is still traversable after 1024 bytes, so part 1 never sees it.